# 05 朴素贝叶斯 Naive Bayes

朴素贝叶斯常用于文本分类、垃圾邮件识别和简单高维分类。它的核心假设很强：给定类别后，各特征条件独立。


## 0. 学习目标和阅读地图

朴素贝叶斯非常适合理解“生成式分类器”的思想。你需要掌握：

1. 先验概率 `P(y)` 和似然 `P(x|y)` 的含义。
2. 为什么文本分类里常用词袋表示。
3. 条件独立假设为什么“朴素”但仍然有效。
4. Laplace smoothing 为什么不可少。


## 1. 数学逻辑

贝叶斯公式：

$$P(y|x)=\frac{P(x|y)P(y)}{P(x)}$$

分类时 `P(x)` 对所有类别一样，可以忽略：

$$\hat y = \arg\max_y P(y)P(x|y)$$

朴素假设把联合概率拆开：

$$P(x|y)=\prod_j P(x_j|y)$$

为了避免很多小概率相乘下溢，实际计算通常取 log：

$$\log P(y|x) \propto \log P(y) + \sum_j \log P(x_j|y)$$


## 1.1 推导拆开看：为什么可以忽略 P(x)

贝叶斯公式是：

$$P(y|x)=\frac{P(x|y)P(y)}{P(x)}$$

分类时，我们只需要比较哪个类别概率最大。对同一个样本 `x`，分母 `P(x)` 对所有类别一样，所以：

$$\arg\max_y P(y|x)=\arg\max_y P(x|y)P(y)$$

朴素贝叶斯进一步假设词之间条件独立：

$$P(x|y)=P(w_1,w_2,\cdots,w_m|y)=\prod_j P(w_j|y)$$

这个假设现实中不完全成立，但它让高维文本分类变得非常简单。


In [ ]:
import numpy as np
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

texts = [
    'win money now', 'cheap prize win', 'limited offer money', 'claim your prize',
    'meeting schedule today', 'project update today', 'please review document', 'team lunch schedule',
    'win cash prize', 'cheap money offer', 'document review meeting', 'project schedule update'
]
labels = np.array([1,1,1,1,0,0,0,0,1,1,0,0])  # 1=spam, 0=normal


## 1.2 文本如何变成特征

原始输入是字符串，例如 `win money now`。`CountVectorizer` 会把它变成词频向量：

- 每一列对应词表里的一个词。
- 每一行对应一条文本。
- 单元格表示该词在该文本中出现多少次。

MultinomialNB 假设这些词频来自某个类别下的多项分布。


In [ ]:
# 从零实现：Multinomial Naive Bayes 的核心计算
vocab = sorted(set(word for text in texts for word in text.split()))
word_to_id = {w: i for i, w in enumerate(vocab)}

alpha = 1.0  # Laplace smoothing，避免未见过的词概率为 0
classes = sorted(set(labels))
class_log_prior = {}
word_log_prob = {}

for c in classes:
    docs_c = [texts[i] for i in range(len(texts)) if labels[i] == c]
    class_log_prior[c] = np.log(len(docs_c) / len(texts))
    counts = np.zeros(len(vocab))
    for text in docs_c:
        for word in text.split():
            counts[word_to_id[word]] += 1
    probs = (counts + alpha) / (counts.sum() + alpha * len(vocab))
    word_log_prob[c] = np.log(probs)

def predict_text(text):
    scores = {}
    for c in classes:
        score = class_log_prior[c]
        for word in text.split():
            if word in word_to_id:
                score += word_log_prob[c][word_to_id[word]]
        scores[c] = score
    return max(scores, key=scores.get), scores

for text in ['win money prize', 'project meeting review']:
    pred, scores = predict_text(text)
    print(text, '->', 'spam' if pred == 1 else 'normal', scores)


## 1.3 从零实现代码怎么读

从零版本的核心步骤：

1. 建立词表 `vocab`。
2. 统计每个类别下每个词出现了多少次。
3. 加 `alpha` 做平滑，避免概率为 0。
4. 对每个类别计算 log prior 和 log likelihood。
5. 对新文本，把包含词的 log 概率加起来，选最大类别。

因为使用 log，乘法变成加法，数值更稳定。


In [ ]:
# sklearn 实战：CountVectorizer + MultinomialNB 是文本分类常见组合
X_train, X_test, y_train, y_test = train_test_split(texts, labels, random_state=42, stratify=labels)
vectorizer = CountVectorizer()
X_train_counts = vectorizer.fit_transform(X_train)
X_test_counts = vectorizer.transform(X_test)

model = MultinomialNB(alpha=1.0)
model.fit(X_train_counts, y_train)
pred = model.predict(X_test_counts)

print('词表:', vectorizer.get_feature_names_out())
print(classification_report(y_test, pred, target_names=['normal', 'spam']))


In [ ]:
# 诊断：查看哪些词更偏向 spam 或 normal
feature_names = vectorizer.get_feature_names_out()
log_prob = model.feature_log_prob_
spam_minus_normal = log_prob[1] - log_prob[0]
order = np.argsort(spam_minus_normal)

print('更偏 normal 的词:')
for i in order[:5]:
    print(f'  {feature_names[i]:>10s}: log_prob_diff={spam_minus_normal[i]: .3f}')

print('\n更偏 spam 的词:')
for i in order[-5:][::-1]:
    print(f'  {feature_names[i]:>10s}: log_prob_diff={spam_minus_normal[i]: .3f}')


## 2.1 如何解释朴素贝叶斯

朴素贝叶斯的一个优点是容易检查词和类别的关系。某个词在 spam 类里的条件概率明显高于 normal 类，就会把预测推向 spam。

但解释时要注意：模型看到的是词频，不理解语义、否定和上下文。例如 `not cheap` 可能仍然因为 `cheap` 被推向 spam。


## 2. 常见误区

- “朴素”指条件独立假设很强，不代表模型没用；文本高维稀疏场景下它常常很强。
- 没有平滑时，测试文本出现训练中未见组合，概率可能被压成 0。
- 词袋模型忽略语序，所以它不能理解复杂句法。

## 3. 小实验

- 改 `alpha`，观察平滑强弱。
- 加入更多容易混淆的文本。
- 把词频换成 TF-IDF，再比较效果。


## 5. 复习清单

- 朴素贝叶斯是生成式分类器。
- 文本分类中常用 MultinomialNB。
- 平滑解决未见词概率为 0 的问题。
- 条件独立假设很强，但在高维稀疏文本上常常有效。
